# Transformations page (draft): demo on the running example

Concept: `overview/concepts/Transfromations/Lineage (DRAFT).md`, section "Transformation Type Level Model", and the playground's transformations overview (outside this repo).
Code: `ocelescope_module_location_based_ocpm.transformations` (`events`, `key_numbers`, `charts`, `profile`, `type_graph`).
Documentation: `docs/pages/transformations.md`.

The page reads the transformation events $T$ and the consumed and produced handling units $cons$, $prod$ of the derivation (`docs/resources/transformation-inputs.md`).

In [ ]:
import json
from pathlib import Path

import polars as pl
from ocelescope import OCEL

from ocelescope_module_location_based_ocpm.derivation.config import DerivationConfig
from ocelescope_module_location_based_ocpm.derivation.pipeline import derive
from ocelescope_module_location_based_ocpm.transformations.charts import transformation_bar_chart
from ocelescope_module_location_based_ocpm.transformations.events import (
    TransformationFilter,
    filter_events,
    transformation_events,
)
from ocelescope_module_location_based_ocpm.transformations.key_numbers import key_numbers
from ocelescope_module_location_based_ocpm.transformations.profile import type_profile
from ocelescope_module_location_based_ocpm.transformations.type_graph import (
    event_sides,
    min_frequency,
    recipe_arcs,
    recipe_keys,
    recipe_table,
    type_graph,
)

EXAMPLE = Path("..") / "data" / "running_example"
INPUTS = EXAMPLE / "inputs"
pl.Config.set_tbl_width_chars(200)
pl.Config.set_tbl_cols(20)

## 1. Transformation events

`derive` gives the roles (one row per consumed or produced handling unit) and the events.
`transformation_events` turns them into one row per transformation event: the input $in(e)$ and output $out(e)$ as labels ("1 slab", "2 coil"; "∅" for an empty side), the object types and quantities per side, the **transformation type** "input → output", the **recipe** "activity: input → output" and the **kind** (both, consumption only, production only).
With `relations` and a location column it also lists the event's locations: the known locations of its handling units after the event (here $loc^3$).

In [ ]:
ocel = OCEL.read(EXAMPLE / "running_example.sqlite")
config = DerivationConfig.model_validate(json.loads((EXAMPLE / "configuration.json").read_text()))
tables = {path.name: pl.read_csv(path, infer_schema=False) for path in INPUTS.glob("*.csv")}
derivation = derive(ocel, config, tables)
derivation.roles.sort("event", "role", "hu")

In [ ]:
table = transformation_events(
    derivation.events, derivation.roles, derivation.handling_units, derivation.relations, derivation.loc
)
table.select("event", "activity", "input", "output", "transformation_type", "recipe", "kind", "locations")

## 2. Page filters

`TransformationFilter` has three filters: "consumes" and "produces" (object types) and "kind".
`filter_events` keeps whole events (AND across filters, OR within one), so a transformation type or recipe is never cut.
Example: consumes slab keeps e3 only.

In [ ]:
filter_events(table, TransformationFilter(consumes=["slab"])).select("event", "recipe")

## 3. Key numbers, charts and profile (as in the playground)

`key_numbers` counts the kept events: transformations = consumed handling units + handling units produced at events without consumption, lineage relations = $\sum_e |cons(e)| \cdot |prod(e)|$, and the shares of handling units produced or consumed at least once (of those with at least one event).

In [ ]:
key_numbers(table, derivation.roles, derivation.relations["hu"].n_unique())

`transformation_bar_chart` draws one of four charts (activities, transformation types, locations, unique counts), each counting distinct events; an event at several locations counts once at each.
Example: the locations chart by activity.

In [ ]:
chart = transformation_bar_chart(table, "locations", "activity")
[(bar.label, bar.count, [(s.value, s.count) for s in bar.segments]) for bar in chart.bars]

`type_profile`: one row per transformation type with its kind, events, transformations, lineage relations and activities (each activity of a type is one recipe of it).

In [ ]:
pl.DataFrame([row.model_dump(exclude={"activities"}) for row in type_profile(table)])

## 4. Transformation type graph

The running example has one event per recipe, so this section also uses a small extended example: a second roll producing 3 coils (e30), a weld of 2 coils into 1 (e31) and a scrap (e32, consumption only).

In [ ]:
events = pl.DataFrame(
    {
        "event": ["e1", "e3", "e15", "e30", "e31", "e32"],
        "activity": ["cast slab", "roll", "cut", "roll", "weld", "scrap"],
        "is_transformation": [True] * 6,
    }
)
roles = pl.DataFrame(
    [
        ("e1", "s1", "produced"), ("e3", "s1", "consumed"), ("e3", "c1", "produced"), ("e3", "c2", "produced"),
        ("e15", "c2", "consumed"), ("e15", "c4", "produced"), ("e30", "s2", "consumed"), ("e30", "c6", "produced"),
        ("e30", "c7", "produced"), ("e30", "c8", "produced"), ("e31", "c6", "consumed"), ("e31", "c7", "consumed"),
        ("e31", "c9", "produced"), ("e32", "c8", "consumed"),
    ],
    schema=["event", "hu", "role"],
    orient="row",
)
hus = pl.DataFrame({"hu": ["s1", "s2", "c1", "c2", "c4", "c6", "c7", "c8", "c9"], "object_type": ["slab"] * 2 + ["coil"] * 7})
extended = transformation_events(events, roles, hus)
extended.select("event", "recipe", "kind")

Steps 1 to 3: `event_sides` gives one row per (event, side, object type) with the quantity; `recipe_keys` the recipe of every event, either **exact** (quantities are part of the recipe, as in the concept) or **as sets** (quantities ignored); `recipe_table` the frequency of every recipe (ids r1, r2, … by frequency); `recipe_arcs` per recipe, side and object type the smallest and largest quantity and the handling units that flowed along the arc.

In [ ]:
sides = event_sides(extended)
keys = recipe_keys(extended, "exact")
recipe_table(keys)

In [ ]:
keys_as_sets = recipe_keys(extended, "sets")
recipe_table(keys_as_sets)

In [ ]:
recipe_arcs(sides, keys_as_sets)

Step 4, the frequency slider: `min_frequency` ranks the recipes by frequency and takes the most frequent ones until they cover the slider's share of the events; every recipe as frequent as the last one taken is shown too, so the slider amounts to a minimum frequency.
With the recipes as sets (frequencies 2, 1, 1, 1, 1; 6 events), 30 % needs 1.8 events, which "roll" (2) covers: minimum frequency 2.

In [ ]:
frequencies = recipe_table(keys_as_sets)["frequency"]
[(share, min_frequency(frequencies, share)) for share in (0.0, 0.3, 0.5, 1.0)]

Step 5: `type_graph` gives the nodes (recipes, object types) and arcs of the shown recipes, which the page lays out with ELK.

In [ ]:
graph = type_graph(extended, coverage=0.3, quantities="sets")
graph.recipes, graph.arcs, (graph.recipes_total, graph.events_shown, graph.events_total, graph.min_frequency)

In [ ]:
full = type_graph(table)
pl.DataFrame([arc.model_dump() for arc in full.arcs])